# Accessing Web Archive Data

<img src="https://ndha-public-data-ap-southeast-2.s3.ap-southeast-2.amazonaws.com/iPRES-2025/resources/nlnz-webarchive-public-portal.png" alt="NLNZ Selective Web Archive portal" border="0">

## Overview

This notebook demonstrates how to access and query web archive data from several open web archives. It provides a foundation for working with web archives using various protocols and APIs.

### Learning Objectives

1. Query web archive data using the Memento protocol
2. Retrieve and interpret capture histories using TimeMaps
3. Access archived content using the CDX API
4. Identify archived content types for analysis

This notebook serves as an introduction to web archive access methods that will be built upon in subsequent notebooks.

## Environment Setup

### Installing Required Python Packages

The following packages are necessary for working with web archives:

In [ ]:
# Colab bootstrap: install the packages required for exploring web archive data in this notebook. You can skip this cell if you are running the notebook in your local environment and have already installed the required packages.
%pip install --upgrade --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ wa-nlnz-toolkit

In [ ]:
# Import the NLNZ Web Archive Toolkit
import wa_nlnz_toolkit as want
import pandas as pd
import datetime
from bs4 import BeautifulSoup

### Configuring Web Archive Endpoints

By default this notebook is configured to query the New Zealand Web Archive. To override this configuration, set the following variables:

In [ ]:
# New Zealand Web Archive
# want.set_memento_url("https://ndhadeliver.natlib.govt.nz/webarchive")

# Norwegian Web Archive
# want.set_memento_url("https://nettarkivet.nb.no/search")

# Australian Web Archive
# want.set_memento_url("https://web.archive.org.au/awa")

# Arquivo.pt
# want.set_memento_url("https://arquivo.pt/wayback")

# Internet Archive
want.set_memento_url("https://web.archive.org/web")

## 1. Querying Web Archives with the Memento Protocol

### Introduction to Memento

![image](https://ndha-public-data-ap-southeast-2.s3.ap-southeast-2.amazonaws.com/iPRES-2025/resources/memento.png)

The **Memento protocol** provides a standardized way to access archived versions of web pages across different web archives. It offers machine-readable information about web captures and simplifies the process of finding historical versions of web content.

### Key Memento Components

The NLNZ web archive supports three main Memento features:

1. **TimeGate** - Retrieves the version of a page closest to a specified date
2. **TimeMap** - Provides a list of all archived versions of a page
3. **Memento** - Represents a specific archived version with options to control presentation

Let's explore how to use these features with the NLNZ Web Archive Toolkit.

### Basic Memento Queries

We'll start by querying the latest capture of a website using the Memento protocol.<br>
Change which URI to query by adding/removing `# ` before one of the lines with `webpage = "<uri>"` examples, or edit the uri within `" "`.<br><br>

In [ ]:
# Define target website

# New Zealand example
# webpage = "https://natlib.govt.nz/"

# Norwegian example
# webpage = "https://www.yr.no/"

# Australian example
# webpage = "https://cobb.qm.qld.gov.au/"

# Portuguese example
# webpage = "https://www.bbc.co.uk"
# webpage = "https://www.abola.pt"

webpage = "https://atlantafwc26.com/"

# Query the latest capture using Memento
# This returns the raw response headers
dict(want.query_memento(webpage).headers)

*(If you're getting an HTTPError 404, it means the archive's Memento API couldn't resolve it.)*

In [ ]:
# Get a more structured representation of Memento URLs
want.get_memento_urls(webpage)

### Understanding Memento Link Types

The Memento response contains several important link types:

- **original**: The original URL that was archived (e.g., https://covid19.govt.nz/)
- **timegate**: The URL used to request archived versions (e.g., https://ndhadeliver.natlib.govt.nz/webarchive/https://covid19.govt.nz/)
- **timemap**: URL that lists all available captures (e.g., https://ndhadeliver.natlib.govt.nz/webarchive/timemap/link/https://covid19.govt.nz/)
- **memento**: URL of the specific archived version (e.g., https://ndhadeliver.natlib.govt.nz/webarchive/20250728214105mp_/https://covid19.govt.nz/)

By default, the *memento* link points to the latest capture. We can also request a capture closest to a specific date:

In [ ]:
# Query for a capture closest to January 1, 2020
dt_required = datetime.datetime(2020, 1, 1, 0, 0, 0)
dict(want.query_memento(webpage, dt=dt_required).headers)

In [ ]:
# Get structured Memento URLs for a specific date
want.get_memento_urls(webpage, dt=dt_required)

In [ ]:
# Query another website to see its Memento links
want.query_memento("www.niwa.co.nz").links

>💡 <strong>Try it:</strong> The memento URL is a clickable link that you can open in a browser to see the archived version.

### Retrieving Complete Capture History with TimeMap

The Memento TimeMap provides a comprehensive list of all captures for a given webpage. The NLNZ web archive supports multiple TimeMap formats (link, cdxj, and json).

Let's retrieve the TimeMap for our example website:

In [ ]:
# Get the TimeMap for the National Library website

# New Zealand Web Archive
# webpage = "www.natlib.govt.nz"

# Norwegian Example (rendering table currently fails, but you'll get a URI for the JSON)
# webpage = "www.yr.no"

# Australian Web Archive
# webpage = "cobb.qm.qld.gov.au/"

# Arquivo.pt
# webpage = "https://www.abola.pt"

webpage = "https://atlantafwc26.com/"

want.get_timemap(webpage)

>💡 <strong>Try it:</strong> Copy the oldest and most recent memento URLs (from the *access_url* column) and paste them into a web browser, to compare the visual difference between the harvests.

### URL Modifiers in Memento

Memento supports special URL modifiers that control how archived content is presented:

- **mp_** modifier: Shows "main page" content only
- **id_** modifier: Returns the original harvested version without rewriting
- **if_** modifier: Shows the page with web archive headers (default for NLNZ web archive)

For more details on URL rewriting options, see the [PyWB documentation](https://pywb.readthedocs.io/en/latest/manual/rewriter.html?highlight=id_#url-rewriting).

## 2. Querying Web Archives with the CDX API

![image](https://ndha-public-data-ap-southeast-2.s3.ap-southeast-2.amazonaws.com/iPRES-2025/resources/outback-cdx.png)

The CDX (Capture inDeX) API provides a more direct way to query web archive metadata. It allows for more specific filtering and returns structured data about archived captures.

> ***Note***
>
> *In the NLNZ web archive, CDX API queries are routed via the PyWB viewer. This means some native CDX query parameters (like output format) are not supported.*

In [ ]:
# Query the CDX index for the National Library website

# New Zealand Web Archive
# webpage = "www.natlib.govt.nz"

# Norwegian Web Archive
# want.set_cdx_api_url("https://nettarkivet.nb.no/search/cdx")
# webpage = "www.yr.no"

# Australian Web Archive
# webpage = "cobb.qm.qld.gov.au/"

# Arquivo.pt
# webpage = "https://www.abola.pt"

# Internet Archive - uses a different cdx endpoint to its memento service
want.set_cdx_api_url("https://web.archive.org/cdx/search/cdx")
webpage = "https://atlantafwc26.com/"

df_captures = want.query_cdx_index(webpage)
df_captures

### CDX vs TimeMap

The CDX query results are similar to the TimeMap. ***Note***, our toolkit is appending an `access_url` column that contains the actual URL for accessing each webpage snapshot. This makes it easier to view or analyze specific captures.

### Advanced CDX Queries

The CDX API allows for more specific queries, such as filtering by MIME type or using prefix matching. This is particularly useful for finding non-HTML content like images or documents.

In [ ]:
# Query for PDF files in a specific section of a website
# Note: Due to architecture limitations, we need to specify at least the first-level path segment (e.g. /assets/)
webpage = "covid19.govt.nz/assets/"

# Norwegian example (no need for path segment)
# webpage = "regjeringen.no"

# Filter for PDF files using the MIME type filter
df_captures = want.query_cdx_index(
    webpage, filter="mimetype:application/pdf", matchType="prefix"
)

# Extract original filenames from the URLs
df_captures["original_file_name"] = df_captures["urlkey"].str.split("/").str[-1]
df_captures

>💡 <strong>Hands-On Exercise:</strong> Querying for Image Files
>
> Try completeing the code below and querying the CDX index for PNG image files from the same website section. What other MIME types can you find?
>
> See the *Hands-On Exercise Solutions* section at the end for a working solution

In [ ]:
# Exercise: Query for PNG files
webpage = "covid19.govt.nz/assets/"

# 1. Query the CDX index, fitlering on the MIME type
# 2. Extract original filenames from the URLs
# 3. Output the results

## Conclusion and Next Steps

This notebook has introduced the fundamental methods for accessing and working with open web archives:

1. **Memento Protocol** - For standardized access to archived web content
2. **TimeMaps** - For listing all mementos of an archived resource
3. **CDX API** - For querying and filtering archive metadata

### What's Next?

In the following notebooks, we'll build on these foundations to:

- Explore and analyze web archive data in more depth
- Track changes in websites over time
- Extract and analyze textual content at scale
- Build advanced applications using web archive data

These techniques provide powerful tools for researchers, historians, and data scientists working with web archives.

---

## Hands-On Exercise Solutions

#### Querying for Image Files

```
# Exercise: Query for PNG files
webpage = "covid19.govt.nz/assets/"

df_captures = want.query_cdx_index(webpage, filter="mimetype:image/png", matchType="prefix")
df_captures["original_file_name"] = df_captures["urlkey"].str.split("/").str[-1]
df_captures
```

### Citation

**Title:** Accessing Web Archive Data (Jupyter notebook)  
**Author:** Yizhe Zhan, Ben O'Brien  
**Affiliation:** National Library of New Zealand, Wellington  
**Last updated:** 2026‑04‑14  

**Contact:** yizhe.git@gmail.com  
**Repository:** https://github.com/NLNZDigitalPreservation/wa-nlnz-toolkit